In [1]:
soc_reviewer_RF=[0.5, 0.5, 0.5257692307692308, 0.7635348936170213, 1.0, 1.0, 1.0, 1.0, 0.7369230769230769, 0.4738461538461539, 0.21076923076923076, 0.21076923076923076, 0.21076923076923076, 0.21076923076923076, 0.21076923076923076, 0.21076923076923076, 0.21076923076923076, 0.21076923076923076, 0.21076923076923076, 0.21076923076923076, 0.21076923076923076, 0.21076923076923076, 0.0, 0.0, 0.0]
# soc_reviewer_LSTM= [0.5, 0.23690312030946836, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.23690312030946836]
soc_reviewer_LSTM= [0.5, 0.23690312030946836, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,0.0]
# soc_reviewer_TLLM = [0.5, 0.5, 0.5, 0.5, 0.7374449339207048, 0.9748898678414096, 1.0, 1.0, 1.0, 0.7369330153385809, 0.4738660306771618, 0.2107990460157427, 0.2107990460157427, 0.4482440009364476, 0.6856889648571525, 0.9231339287778573, 1.0, 1.0, 0.7369330153385809, 0.4738660306771618, 0.2107990460157427, 0.0, 0.0, 0.0,0.0]
soc_reviewer_TLLM = [0.5, 0.5, 0.5, 0.5, 0.7374449339207048, 0.9748898678414096, 1.0, 1.0, 1.0, 0.7369330153385809, 0.4738660306771618, 0.2107990460157427, 0.2107990460157427, 0.4482440009364476, 0.6856889648571525, 0.9231339287778573, 1.0, 0.7369330153385809, 0.4738660306771618, 0.2107990460157427, 0.0, 0.0, 0.0,0.0,0.0]

In [2]:
len(soc_reviewer_RF), len(soc_reviewer_LSTM), len(soc_reviewer_TLLM)

(25, 25, 25)

In [3]:
import pandas as pd

data_MILP = pd.read_csv("outputs/MILP_actual_perfect.csv")
data_MILP.head()

,prices_actual,prices_forecast,actual_demand,forecast_demand,soc,charge_MW,discharge_MW,import_MW,export_MW,profit_step
0,65.10,65.10,28.68,28.68,0.50000,0.00,0.0,28.68,0.0,0.0000
1,62.12,62.12,27.12,27.12,0.50000,0.00,0.0,27.12,0.0,0.0000
2,59.05,59.05,25.99,25.99,0.50000,0.00,0.0,25.99,0.0,0.0000
3,59.00,59.00,25.62,25.62,0.50000,1.14,0.0,26.76,0.0,-67.2600
4,56.63,56.63,25.66,25.66,0.52511,10.78,0.0,36.44,0.0,-610.4714


In [4]:
data_version = "forecast"
forecast_type = "RF"
duration_hours=4
capacity_MWh = duration_hours*10.78
soc_init=0.5
soc_min=0.0
soc_max=1.0
eta_c = 0.95
eta_d = 0.95
soc_target=0.5

In [5]:
import warnings
warnings.filterwarnings("ignore")  # Ignore all warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from agentic_energy.schemas import BatteryParams, DayInputs, SolveRequest
battery=BatteryParams(
    capacity_MWh=capacity_MWh,
    cmax_MW=10.78,
    dmax_MW=10.78,
    soc_init=soc_init,
    soc_min=soc_min,
    soc_max=soc_max,
    eta_c=eta_c,
    eta_d=eta_d,
    soc_target=soc_target,
)
battery

BatteryParams(capacity_MWh=43.12, soc_init=0.5, soc_min=0.0, soc_max=1.0, cmax_MW=10.78, dmax_MW=10.78, eta_c=0.95, eta_d=0.95, soc_target=0.5)

In [6]:
from typing import Sequence, Optional
def cost_from_soc(
    soc: Sequence[float],
    prices_buy: Sequence[float],
    demand_MW: Sequence[float],
    *,
    battery: BatteryParams,
    prices_sell: Optional[Sequence[float]] = None,
    allow_export: bool = False,
    dt_hours: float = 1.0,
):
    soc = np.asarray(soc, dtype=float)
    assert len(soc) >= 2, "SOC must include at least t=0 and t=1"
    T = len(soc) - 1

    prices_buy  = np.asarray(prices_buy, dtype=float)
    demand_MW   = np.asarray(demand_MW, dtype=float)
    assert len(prices_buy) == T and len(demand_MW) == T

    if prices_sell is None:
        prices_sell = prices_buy
    prices_sell = np.asarray(prices_sell, dtype=float)
    assert len(prices_sell) == T

    # Per-step energy change in MWh
    dE = (soc[1:] - soc[:-1]) * battery.capacity_MWh

    # Recover charge/discharge MW from SOC deltas and efficiencies
    charge_MW    = np.maximum(dE, 0.0) / (battery.eta_c * dt_hours)
    discharge_MW = np.maximum(-dE, 0.0) * (battery.eta_d / dt_hours)

    # Enforce hardware limits
    charge_MW    = np.minimum(charge_MW,    battery.cmax_MW)
    discharge_MW = np.minimum(discharge_MW, battery.dmax_MW)

    # Grid net load
    net = demand_MW + charge_MW - discharge_MW
    imp = np.maximum(net, 0.0)
    exp = np.maximum(-net, 0.0) if allow_export else np.zeros_like(net)

    # Cost (buy imports, optionally credit exports)
    cost = float(np.sum(prices_buy * imp * dt_hours) - np.sum(prices_sell * exp * dt_hours))

    out = {
        "charge_MW": charge_MW,
        "discharge_MW": discharge_MW,
        "import_MW": imp,
        "export_MW": exp,
        "net_MW": net,
        "objective_cost": cost,
    }
    return out

In [7]:
out_RF = cost_from_soc(
    soc = soc_reviewer_RF,
    prices_buy=data_MILP["prices_actual"].values,
    demand_MW=data_MILP["actual_demand"].values,
    battery=battery,
    prices_sell=data_MILP["prices_actual"].values,
    allow_export=True,
    dt_hours=1
)

out_LSTM = cost_from_soc(
    soc = soc_reviewer_LSTM,
    prices_buy=data_MILP["prices_actual"].values,
    demand_MW=data_MILP["actual_demand"].values,
    battery=battery,
    prices_sell=data_MILP["prices_actual"].values,
    allow_export=True,  
    dt_hours=1
)

out_TLLM = cost_from_soc(
    soc = soc_reviewer_TLLM,
    prices_buy=data_MILP["prices_actual"].values,
    demand_MW=data_MILP["actual_demand"].values,
    battery=battery,
    prices_sell=data_MILP["prices_actual"].values,
    allow_export=True,  
    dt_hours=1
)


In [8]:
# calculate profit as (discharge_MW - charge_MW)*prices_actual
profit_RF = np.sum(out_RF["discharge_MW"] * data_MILP["prices_actual"].values) - np.sum(out_RF["charge_MW"] * data_MILP["prices_actual"].values)
profit_LSTM = np.sum(out_LSTM["discharge_MW"] * data_MILP["prices_actual"].values) - np.sum(out_LSTM["charge_MW"] * data_MILP["prices_actual"].values)
profit_TLLM = np.sum(out_TLLM["discharge_MW"] * data_MILP["prices_actual"].values) - np.sum(out_TLLM["charge_MW"] * data_MILP["prices_actual"].values)

print(f"Profit from RF reviewer SOC: {profit_RF:.2f}")
print(f"Profit from LSTM reviewer SOC: {profit_LSTM:.2f}")
print(f"Profit from TLLM reviewer SOC: {profit_TLLM:.2f}")  

Profit from RF reviewer SOC: 1578.52
Profit from LSTM reviewer SOC: 1304.46
Profit from TLLM reviewer SOC: 2030.46


In [9]:
data_MILP["profit_step"].sum()

np.float64(2246.0897118282546)